In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin
import pandas as pd
from datetime import datetime

# Weekly Charts

In [ ]:
# Target URLs Example
# https://kworb.net/spotify/country/global_weekly.html
# https://kworb.net/spotify/country/us_weekly.html

# Get all Area URLs
def get_all_area_urls():
    url = f"https://kworb.net/spotify/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # Define Agent
    html = requests.get(url, headers=headers, timeout=20)
    html.raise_for_status()
    soup = BeautifulSoup(html.content, "html.parser")

    # Find required table
    table = soup.select_one("table[style*='width: 410px']")
    if not table:
        return []
    
    # Retrieve all weekly links (Except *_weekly_totals.html)
    results = []
    for tr in table.select("tr"):
        country_td = tr.select_one("td:nth-of-type(1)")
        weekly_a = tr.select_one("a[href$='_weekly.html']")
        country = country_td.get_text(strip=True)
        weekly_url = urljoin(url, weekly_a["href"])

        results.append({"country": country, "url": weekly_url})

    return results

all_area_urls = pd.DataFrame(get_all_area_urls())

In [ ]:
# Get Weekly Chart from Certain Area
def get_weekly_chart(url: str, limit=5):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    html = requests.get(url, headers=headers, timeout=20)
    html.raise_for_status()
    soup = BeautifulSoup(html.content, "html.parser")

    # Retrieve update date
    chart_date = None
    title_el = soup.select_one("span.pagetitle")
    if title_el:
        title_text = title_el.get_text(" ", strip=True)
        m = re.search(r"\b(\d{4}/\d{2}/\d{2})\b", title_text)
        if m:
            chart_date = datetime.strptime(m.group(1), "%Y/%m/%d").date()

    # Retrieve Weekly Chart Info
    table = soup.select_one("table#spotifyweekly")
    if not table:
        return []

    chart_data = []
    for row in table.select("tbody tr")[:limit]:
        rank = row.select_one("td:nth-of-type(1)").get_text(strip=True)
        artist = row.select_one("td:nth-of-type(3) a[href*='../artist/']").get_text(strip=True)
        title = row.select_one("td:nth-of-type(3) a[href*='../track/']").get_text(strip=True)
        streams = row.select_one("td:nth-of-type(7)").get_text(strip=True)
        chart_data.append({"rank": rank, "artist": artist, "title": title, "streams": streams, "chart_date": chart_date, "chart_url": url})

    return chart_data

In [ ]:
# Scrape all areas weekly charts
def scrape_all_weekly_charts(all_area_urls: pd.DataFrame, limit: int = 20) -> pd.DataFrame:
    all_rows = []
    total = len(all_area_urls)

    # Iterate
    for i, (_, r) in enumerate(all_area_urls.iterrows(), start=1):
        country = r["country"]
        url = r["url"]
        remaining = total - i

        print(f"[{i}/{total}] Scraping {country}... "
              f"({remaining} urls remaining)")

        # Use get_weekly_chart in each area
        chart = get_weekly_chart(url, limit=limit)

        # Add needed infos
        for item in chart:
            item["country"] = country
            all_rows.append(item)

    print("Scraping completed")

    return pd.DataFrame(all_rows)

weekly_chart_df = scrape_all_weekly_charts(all_area_urls, limit=20)

In [ ]:
weekly_chart_df

In [ ]:
weekly_chart_df[weekly_chart_df["country"] == "Taiwan"]

In [ ]:
# Convert streams to float
df = weekly_chart_df.copy()
df['streams'] = df['streams'].str.replace(',', '').astype(float)

In [ ]:
streams_df = df.groupby('country')['streams'].sum().reset_index() # sum of streams 
streams_df = streams_df.drop(24, axis=0) # drop global row
streams_df = streams_df.reset_index(drop=True)

In [ ]:
tmp = streams_df.sort_values('streams', ascending=False).reset_index()
tmp = tmp.drop('index', axis=1)
tmp

## Country Users Map

In [ ]:
import folium
import folium.plugins

In [ ]:
# GeoJSON for countries
political_countries_url = ("http://geojson.xyz/naturalearth-3.3.0/ne_50m_admin_0_countries.geojson")

In [ ]:
# Map test (Streams of top 5 songs)
m = folium.Map(location=(30, 10), zoom_start=2, tiles="cartodb positron")
folium.Choropleth(
    geo_data=political_countries_url,
    data=streams_df,
    columns=("country", "streams"),
    key_on="feature.properties.name", # keys to link the data with gejson
    bins=6, 
    fill_color="YlGnBu",
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="white", # nans shall be left white
    legend_name="Streams",
    name="Countries by Weekly Total Streams of 20 Songs",
).add_to(m)
folium.LayerControl().add_to(m)

folium.plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m)

fig = folium.Figure(width = 1000, height = 800)
fig.add_child(m)
m.save("streams_map.html")
m



## Artist Country

In [ ]:
import re
import requests

In [ ]:
def clean_artist_name(name):
    name = re.split(r',|&|feat\.|featuring', name, flags=re.IGNORECASE)[0] # remove features
    return name.strip()

In [ ]:
# API using musicbrainz (caution: 2 second rate limit makes many request slow) 
def get_artist_country(artist_name):

    artist_name = clean_artist_name(artist_name)
    
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    url = "https://musicbrainz.org/ws/2/artist/"
    params = {"query": artist_name, 
              "fmt": "json",
              "limit": 1}

    try:
        time.sleep(2.0)
        response = requests.get(url, params=params, headers=headers)
        data = response.json()

        if data['artists']:
            artist = data['artists'][0]

            if 'country' in artist: # country
                return artist['country']

            area = artist.get('area')
            if isinstance(area, dict):
                return area.get('name', 'Unknown')

    except:
        pass

    return "Unknown"

In [ ]:
get_artist_country('Bad Bunny') # test

In [ ]:
# artist country dictionary
# try for weekly_chart_df
unique_artists = weekly_chart_df['artist'].unique()

artist_country = {}
for artist in unique_artists:
    artist_country[artist] = get_artist_country(artist)

In [ ]:
#artist_country

Next we can try for top 500 artists

In [ ]:
# top 500 artist streams (total) 
def get_artist_top500(url: str, limit=500):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    html = requests.get(url, headers=headers, timeout=20)
    html.raise_for_status()
    soup = BeautifulSoup(html.content, "html.parser")

    # Retrieve Chart Info
    table = soup.select_one("div.container table")
    if not table:
        return []

    chart_data = []
    rows = table.select("tr")
    for row in table.select("tbody tr")[:limit]:
        artist = row.select_one("td:nth-of-type(1)").get_text(strip=True)
        streams = row.select_one("td:nth-of-type(2)").get_text(strip=True)
        chart_data.append({"artist": artist, "total_streams": streams, "chart_url": url})

    chart_data = pd.DataFrame(chart_data)
    return chart_data 

In [ ]:
url = "https://kworb.net/spotify/artists.html"
top500_artist_df = get_artist_top500(url, limit=500)

In [ ]:
top500_artist_df

In [ ]:
# Country dictionary (caution: takes a long time to run)
artist_country = {}
for artist in top500_artist_df['artist']:
    artist_country[artist] = get_artist_country(artist)

In [ ]:
#artist_country 

In [ ]:
top500_artist_df['country'] = top500_artist_df['artist'].map(artist_country)

In [ ]:
top500_artist_df['country'].unique()

In [ ]:
# What are the unknowns
top500_artist_df.loc[top500_artist_df['country'] == 'Unknown']

In [ ]:
# change names to countries

# https://www.iban.com/country-codes (ISO 3166-1 Alpha-2 country codes)
country_rename = {"Scotland": "GB", 
                  "Buenos Aires": "AR", 
                  "McAllen": "US",
                  "Miami": "US",
                  "Hawaii": "US",
                  "Gujarat": "IN", 
                  "England": "GB",
                  "Los Angeles": "US",
                  "Las Palmas de Gran Canaria": "ES",
                  "Sinaloa": "MX",
                  "Guadalajara": "MX",
                  "Mazatlan": "MX",
                  "Culiacán": "MX", 
                  "Punjab": "IN"}

top500_artist_new = top500_artist_df.copy()
top500_artist_new['country'] = top500_artist_new['country'].replace(country_rename)

# Manually change unknowns
top500_artist_new.loc[top500_artist_new['artist'] == 'Mora', 'country'] = 'PR'
top500_artist_new.loc[top500_artist_new['artist'] == 'League of Legends', 'country'] = 'US'
top500_artist_new.loc[top500_artist_new['artist'] == 'Original Broadway Cast of Hamilton', 'country'] = 'US'
top500_artist_new.loc[top500_artist_new['artist'] == 'Bibi und Tina', 'country'] = 'DE'

In [ ]:
top500_artist_new['country'].unique()

In [ ]:
#pd.set_option('display.max_rows', None)
#pd.reset_option('display.max_rows')
top500_artist_new

In [ ]:
top500_artist_countries_count = top500_artist_new['country'].value_counts().reset_index()
top500_artist_countries_count.columns = ['country', 'count']

In [ ]:
# tmp = top500_artist_countries_count
# tmp['country'] = tmp['country'].apply(convert_country)

In [ ]:
# tmp

In [ ]:
type(top500_artist_countries_count['country'])

In [ ]:
# GeoJSON for countries
countries_2code_url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
countries_2code_json = requests.get(countries_2code_url).json()

In [ ]:
countries_2code_json["features"][0]["properties"]

In [ ]:
countries_2code_json.keys()

In [ ]:
# Map (Top 500 artists by country)
m = folium.Map(location=(30, 10), zoom_start=2, tiles="cartodb positron")
folium.Choropleth(
    geo_data=countries_2code_json,
    data=top500_artist_countries_count,
    columns=("country", "count"),
    key_on="properties.ISO3166-1-Alpha-2", # match with the alpha2
    threshold_scale=[1, 5, 10, 30, 50, 245],
    fill_color="YlGnBu",
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="white", # nans shall be left white
    legend_name="Number of Artists",
    name="Countries by Number of Artists in Top 500 Streams",
).add_to(m)
folium.LayerControl().add_to(m)

folium.plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m)

fig = folium.Figure(width = 1000, height = 800)
fig.add_child(m)

fig.save('artists_map.html')
m



## Artist Genres

In [ ]:
# API using musicbrainz (caution: 2 second rate limit makes many request slow) 
def get_artist_genre(artist_name):

    artist_name = clean_artist_name(artist_name)
    
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    url = "https://musicbrainz.org/ws/2/artist/"
    params = {"query": artist_name, 
              "fmt": "json",
              "limit": 1}

    try:
        time.sleep(2.0)
        response = requests.get(url, params=params, headers=headers)
        data = response.json()

        if data['artists']:
            artist = data['artists'][0]

            # music brainz has genres entity so try it out
            genres = artist.get('genres', [])
            if genres:
                # users tag genres to artist so sort by count genre count and take the most common one
                genres_sorted = sorted(genres, key=lambda x: x["count"], reverse=True)
                return genres_sorted[0]["name"].title()

            # many unknowns, try same methodology for "tags" entity
            tags = artist.get("tags", [])
            if tags:
                tags_sorted = sorted(tags, key=lambda x: x["count"], reverse=True)
                return tags_sorted[0]["name"].title()

    except:
        pass

    return "Unknown"

In [ ]:
# test
get_artist_genre('Bad Bunny')

In [ ]:
# apply to our df (caution: takes a long time to run)
artist_genre = {} # dictionary
for artist in top500_artist_df['artist']:
    artist_genre[artist] = get_artist_genre(artist)

In [ ]:
#pd.set_option('display.max_rows', None)
#pd.reset_option('display.max_rows')
top500_artist_new['genre'] = top500_artist_new['artist'].map(artist_genre)
top500_artist_new

In [ ]:
# What are the unknowns
top500_artist_new.loc[top500_artist_new['genre'] == 'Unknown']

In [ ]:
print(top500_artist_new['genre'].unique())

In [ ]:
top500_artist_genre_count = top500_artist_new['genre'].value_counts().reset_index()
top500_artist_genre_count.columns = ['genre', 'count']

In [ ]:
top500_artist_genre_count

In [ ]:
# remove unknown
top_genres = top500_artist_genre_count[top500_artist_genre_count['genre'] != 'Unknown']
top_genres = top_genres.head(20)
top_genres

In [ ]:
# create manual groupings
genre_map = {'Hip Hop': 'Hip Hop', 'Trap': 'Hip Hop', 'Pop Rap': 'Hip Hop',
    'Pop': 'Pop', 'Indie Pop': 'Pop', 'K-Pop': 'Pop',
    'Electronic': 'Electronic', 'Dance-Pop': 'Electronic', 'House': 'Electronic',
    'Latin': 'Latin', 'Reggaeton': 'Latin', 'Trap Latino': 'Latin', 'Latin Pop': 'Latin', 'Regional Mexicano': 'Latin',
    'Rock': 'Rock', 'Indie Rock': 'Rock', 'Alternative Rock': 'Rock',
    'Country': 'Other/Region Genre', 'Filmi': 'Other/Region Genre', 'Contemporary R&B': 'Other/Region Genre'
}
             
            
top_genres['genre_main'] = top_genres['genre'].map(genre_map)

In [ ]:
# Preliminary plot
import plotly.express as px

color_map = {'Hip Hop': '#ff0000',
             'Pop': '#0000ff',
             'Electronic':'#ff00ff',
             'Latin': '#ffff00',
             'Rock': '#00ff00',
             'Other/Region Genre': '#00ffff'           
}  

fig = px.bar(top_genres, 
    x='count', y='genre', title="Top 20 Genres Represented By Top 500 Global Artists",
    color = 'genre_main', color_discrete_map = color_map, orientation='h'
)

fig.update_layout(yaxis={'categoryorder': 'total ascending'},
    height = 600, plot_bgcolor='rgba(0,0,0,0)')

top_genres = top_genres[top_genres['genre'] != 'Global'] # remove global

fig.show()
fig.write_html("top_genres.html")

# Distribution Across Countries

Lets look at top 20 for each country

In [ ]:
weekly_chart_df_20 = scrape_all_weekly_charts(all_area_urls, limit=20)

In [ ]:
# Map artists to country (caution: takes a long time to run)
unique_artists = weekly_chart_df_20['artist'].unique()
artist_country2 = {}
for artist in unique_artists:
    artist_country2[artist] = get_artist_country(artist)

In [ ]:
# Map artists to genre (caution: takes a long time to run)
artist_genre2 = {}
for artist in unique_artists:
    artist_genre2[artist] = get_artist_genre(artist)

In [ ]:
# apply dictionaries to df
weekly_chart_df_20['artist_country'] = weekly_chart_df_20['artist'].map(artist_country2)
weekly_chart_df_20['artist_genre'] = weekly_chart_df_20['artist'].map(artist_genre2)

In [ ]:
#pd.reset_option("display.max_rows")
unique_country = weekly_chart_df_20.loc[weekly_chart_df_20['artist_country'] != 'Unknown', 'artist_country'].drop_duplicates()
unique_country

In [ ]:
len(unique_country)

In [ ]:
# check unknowns
unique_unknowns = weekly_chart_df_20.loc[weekly_chart_df_20['artist_country'] == 'Unknown', 'artist'].drop_duplicates()
len(unique_unknowns)

In [ ]:
# change names to countries

# https://www.iban.com/country-codes (ISO 3166-1 Alpha-2 country codes)
country_rename = {"Scotland": "GB", "Buenos Aires": "AR", "McAllen": "US", "Miami": "US", "Hawaii": "US",
                  "Gujarat": "IN", "England": "GB", "Los Angeles": "US", "Las Palmas de Gran Canaria": "ES",
                  "Sinaloa": "MX", "Guadalajara": "MX", "Mazatlan": "MX", "Culiacán": "MX", "Punjab": "IN",
                  "Florida": "US", "Montgomery": "US", "Lübeck": "DE", "Santiago": "CL", "San Francisco": "US",
                  "Barcelona": "ES", "Karlovy Vary": "CZ", "Copenhagen": "DK", "Helsinki": "FI", "Athens": "GR",
                  "Mumbai": "IN", "Uttarakhand": "IN", "Jammu and Kashmir": "IN", "Genova": "CH", "Napoli": "IT",
                  "Moscow": "RU", "New York": "US", "Paris": "FR", "El Jadida": "MA", "Lagos": "NG", "Port Elizabeth": "ZA",
                  "Nes": "NO", "Nes": "NO", "Karachi": "PK", "Liverpool": "UK", "Davao City": "PH", "Tacloban": "PH", 
                  "Szczecin": "PL", "Warsaw": "PL", "Brooklyn": "US", "Ciudad de México": "MX", "Sevilla": "ES", 
                  "Kaohsiung": "TW", "Kyïv": "UA", "Kerala": "IN", "Montréal": "CA", "Hanoi": "VN", "Ho Chi Minh": "VN",
                  "Lahore": "PK", "Ski": "NO", "Ufa": "RU"}

weekly_chart_df_20_new = weekly_chart_df_20.copy()
weekly_chart_df_20_new['artist_country'] = weekly_chart_df_20_new['artist_country'].replace(country_rename)

In [ ]:
# check
unique_country = weekly_chart_df_20_new.loc[weekly_chart_df_20['artist_country'] != 'Unknown', 'artist_country'].drop_duplicates()
#unique_country 

In [ ]:
# check unknowns
unique_unknown_country = weekly_chart_df_20_new.loc[weekly_chart_df_20_new['artist_country'] == 'Unknown', 'artist'].drop_duplicates()
unique_unknown_genre = weekly_chart_df_20_new.loc[weekly_chart_df_20_new['artist_genre'] == 'Unknown', 'artist'].drop_duplicates()
#unique_unknown_country

The unknowns are for a number of reasons: 
1. user not in database
2. user in database but no associated country/area codes
3. database name and charts name are not matching in language
4. Name overlaps causes database to pick wrong artist.

In [ ]:
len(unique_unknown_country)

In [ ]:
len(unique_unknown_genre)

In [ ]:
len(unique_artists)

In [ ]:
len(unique_country)

In [ ]:
151/581

In [ ]:
300/581

# Listener Distribution Among Countries

Do people listen to artists from domestic or international?

In [ ]:
import pycountry

# convert country names to iso code
def convert_iso(country_name):

    try:
        return pycountry.countries.lookup(country_name).alpha_2
    except: 
        return 'Unknown'

In [ ]:
# convert iso code to country names (will use later for plots)
def convert_country(iso_code):
    try:
        return pycountry.countries.get(alpha_2=iso_code.upper()).name
    except: 
        return 'Unknown'

In [ ]:
# get iso codes
weekly_chart_df_20_new["country_iso"] = weekly_chart_df_20_new["country"].apply(convert_iso)

In [ ]:
#weekly_chart_df_20_new

In [ ]:
type(weekly_chart_df_20_new)

In [ ]:
weekly_chart_df_20_new.info()

In [ ]:
# Define song as domestic, international, or unknown
def artist_locality(row):
    artist_country = row['artist_country']
    chart_country = row['country_iso']

    if artist_country == 'Unknown':
        return 'Unknown'
        
    if artist_country == chart_country:
        return 'Domestic'
    else:
        return 'International'

In [ ]:
weekly_chart_df_20_new["artist_locality"] = weekly_chart_df_20_new.apply(artist_locality, axis=1)

## Artist Locality Percentage Share

In [ ]:
# Change streams to integer from string
weekly_chart_df_20_new['streams'] = weekly_chart_df_20_new['streams'].str.replace(',', '').astype(int)

In [ ]:
# get streams df
streams_df = weekly_chart_df_20_new.groupby(['country', 'artist_locality'])['streams'].sum().unstack(fill_value=0)
streams_df

In [ ]:
streams_df['total_streams'] = streams_df.sum(axis=1) # total number of streams
#streams_df

In [ ]:
# percentages for each category
streams_df['domestic_pct'] = (streams_df['Domestic'] / streams_df['total_streams']) * 100
streams_df['int_pct'] = (streams_df['International'] / streams_df['total_streams']) * 100
streams_df['unknown_pct'] = (streams_df['Unknown'] / streams_df['total_streams']) * 100

In [ ]:
streams_df.head(10)

In [ ]:
streams_df = streams_df.drop('Global')
streams_df = streams_df.reset_index()

In [ ]:
domestic_sort = streams_df.sort_values('domestic_pct', ascending=True)
int_sort = streams_df.sort_values('int_pct', ascending=False)
unknown_sort = streams_df.sort_values('unknown_pct', ascending=False)

#domestic_sort.head(5) #--> 100% highest
#int_sort.head(5)# --> 100% highest
#unknown_sort.head(5)# --> 79% highest

In [ ]:

fig = px.bar(domestic_sort, 
    x=["domestic_pct", "int_pct", "unknown_pct"], y="country", title="Music Consumption by Artist Locality by Country",
    labels={"value": "Percentage (%)", "country": "Country", "variable": "Locality"},
    color_discrete_map={"domestic_pct": "skyblue", "int_pct": "#50C878", "unknown_pct": "grey"},
    orientation='h', height=1500 # Tall height so country names are readable
)

fig.update_layout(
    barmode='stack',
    legend_title_text='Category',
    xaxis_range=[0, 100], # Force 0 to 100%
    hovermode="y unified" # Shows all three values when hovering over a country
)

fig.show()
fig.write_html("artist_locality_interactive.html")

## Manually update country/genre for top artists

In [ ]:
# Country
top_unknowns = weekly_chart_df_20_new[weekly_chart_df_20_new['artist_country'] == 'Unknown']['artist'].value_counts()
#print(top_unknowns[top_unknowns > 1])

In [ ]:
# Manually change top unknowns
update = {
    'Omar Courtz': {'country': 'PR', 'genre': 'Reggaeton'},
    'El Bogueto': {'country': 'MX', 'genre': 'Reggaeton'},
    'HUNTR/X': {'country': 'US', 'genre': 'K-Pop'},
    'Jin': {'country': 'KR', 'genre': 'K-Pop'},
    'HermesHermes': {'country': 'GR', 'genre': 'Hip Hop'},
    'DJ Japa NK': {'country': 'BR', 'genre': 'Funk'},
    'W Sound': {'country': 'AR', 'genre': 'Trap Latino'},
    'V': {'country': 'KR', 'genre': 'K-Pop'},
    'Bella Kay' : {'country': 'US', 'genre': 'Indie Pop'},
    'Mr Plata': {'country': 'CO', 'genre': 'Reggaeton'},
    'Lvbel C5': {'country': 'TR', 'genre': 'Hip Hop'},
    'урал гайсин': {'country': 'RU', 'genre': 'K-Pop'},
    'ARIA VEGA': {'country': 'CO', 'genre': 'R&B'},
    'TUL8TE': {'country': 'EG', 'genre': 'Hip Hop'},
    'Ursaru': {'country': 'RO', 'genre': 'Hip Hop'},
    'Aarne': {'country': 'RO', 'genre': 'Hip Hop'},
    'dabbackwood': {'country': 'RU', 'genre': 'Plugg'},
    'Rambo goyard': {'country': 'FR', 'genre': 'Hip Hop'},
    'Selina': {'country': 'BG', 'genre': 'Pop'},
    'Emanuela': {'country': 'BG', 'genre': 'Chalga'},
    'Mirela': {'country': 'BG', 'genre': 'Pop'},
    'Noah Kahan': {'country': 'US', 'genre': 'Folk Pop'},
    'Jere Klein': {'country': 'CL', 'genre': 'Reggaeton'},
    'Yeison Jimenez': {'country': 'CO', 'genre': 'K-Pop'},
    'YOVNGCHIMI': {'country': 'PR', 'genre': 'Trap Latino'},
    'Çağla': {'country': 'TR', 'genre': 'Pop'},
    'Poizi': {'country': 'TR', 'genre': 'Hip Hop'},
    '4rano': {'country': 'SK', 'genre': 'Pop'},
    'ZIAD ZAZA': {'country': 'EG', 'genre': 'Hip Hop'},
    'La Rvfleuze': {'country': 'FR', 'genre': 'Hip Hop'},
    'Trannos': {'country': 'GR', 'genre': 'Hip Hop'},
    'Navjot Ahuja': {'country': 'IN', 'genre': 'Pop'},
    'Idgitaf': {'country': 'ID', 'genre': 'Indonesian Pop'},
    'Raim Laode': {'country': 'ID', 'genre': 'Indonesian Pop'},
    'Kid Yugi': {'country': 'IT', 'genre': 'Hip Hop'},
    'Ernar Amandyq': {'country': 'KZ', 'genre': 'Indie-Pop'},
    'Mavo': {'country': 'NG', 'genre': 'Afrobeats'},
    'FOLA': {'country': 'NG', 'genre': 'Afrobeats'},
    'fitterkarma': {'country': 'PH', 'genre': 'Rock'},
    'Feza': {'country': 'ZA', 'genre': 'Maskandi'},
    'Ntencane': {'country': 'ZA', 'genre': 'Maskandi'},
    'Sam Deep': {'country': 'ZA', 'genre': 'House'},  
    'BLOK3': {'country': 'TR', 'genre': ' Hip Hop'},
    'Jimin': {'country': 'KR', 'genre': 'K-Pop'},
    'Ezzy R': {'country': 'DR', 'genre': 'Hip Hop'}
}

weekly_chart_df_20_clean = weekly_chart_df_20_new.copy()

for artist, info in update.items():
    weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist'] == artist, ['artist_country', 'artist_genre']] = [info['country'], info['genre']]

In [ ]:
# check unknowns
unique_unknowns_country = weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist_country'] == 'Unknown', 'artist'].drop_duplicates()

In [ ]:
len(unique_unknowns_country)/len(unique_artists)

In [ ]:
pd.set_option('display.max_rows', None)
genre_count = weekly_chart_df_20_clean['artist_genre'].value_counts().reset_index()
genre_count.columns = ['genre', 'count']



In [ ]:
# Genre
top_unknowns = weekly_chart_df_20_clean[weekly_chart_df_20_clean['artist_genre'] == 'Unknown']['artist'].value_counts()
print(top_unknowns[top_unknowns > 2])

In [ ]:
# Fix DR genre
weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist'] == 'Ronny GTA', 'artist_genre'] = 'Latin Urban'


In [ ]:
# Fix HK genre
weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist_genre'] == 'Actor', 'artist_genre'] = 'Cantopop'

In [ ]:
unique_unknowns_genre = weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist_genre'] == 'Unknown', 'artist'].drop_duplicates()

In [ ]:
len(unique_unknowns_genre)/len(unique_artists)

## Redo counts with cleaned df

In [ ]:
weekly_chart_df_20_clean["artist_locality"] = weekly_chart_df_20_clean.apply(artist_locality, axis=1)

In [ ]:
# get streams df
streams_df2 = weekly_chart_df_20_clean.groupby(['country', 'artist_locality'])['streams'].sum().unstack(fill_value=0)

In [ ]:
streams_df2['total_streams'] = streams_df2.sum(axis=1) # total number of streams

In [ ]:
# percentages for each category
streams_df2['domestic_pct'] = (streams_df2['Domestic'] / streams_df2['total_streams']) * 100
streams_df2['int_pct'] = (streams_df2['International'] / streams_df2['total_streams']) * 100
streams_df2['unknown_pct'] = (streams_df2['Unknown'] / streams_df2['total_streams']) * 100

In [ ]:
streams_df2 = streams_df2.reset_index()

In [ ]:
domestic_sort2 = streams_df2.sort_values('domestic_pct', ascending=True)
int_sort2 = streams_df2.sort_values('int_pct', ascending=False)
unknown_sort2 = streams_df2.sort_values('unknown_pct', ascending=False)

#domestic_sort.head(5) #--> 100% highest
#int_sort.head(5)# --> 100% highest
#unknown_sort.head(5)# --> 33% highest

In [ ]:
import plotly.express as px

fig = px.bar(domestic_sort2, 
    x=["domestic_pct", "int_pct", "unknown_pct"], y="country", title="Music Consumption by Artist Locality by Country Cleaned Dataset",
    labels={"value": "Percentage (%)", "country": "Country", "variable": "Locality"},
    color_discrete_map={"domestic_pct": "skyblue", "int_pct": "#50C878", "unknown_pct": "grey"},
    orientation='h', height=1500 # Tall height so country names are readable
)

fig.update_layout(
    barmode='stack',
    legend_title_text='Category',
    xaxis_range=[0, 100], # Force 0 to 100%
    hovermode="y unified" # Shows all three values when hovering over a country
)

fig.show()
fig.write_html("artist_locality_interactive_clean.html")

## Domestic vs International listening Map


In [ ]:
streams_df2["country_iso"] = streams_df2["country"].apply(convert_iso) # get iso codes

In [ ]:
# manually fix Russia and Turkey
streams_df2.loc[streams_df2['country'] == 'Russia', 'country_iso'] = 'RU'
streams_df2.loc[streams_df2['country'] == 'Turkey', 'country_iso'] = 'TR'

In [ ]:
# Map (Domestic Percentage)
m = folium.Map(location=(30, 10), zoom_start=2, tiles="cartodb positron")
folium.Choropleth(
    geo_data=countries_2code_json, # same json as before
    data=streams_df2,
    columns=("country_iso", "domestic_pct"),
    key_on="properties.ISO3166-1-Alpha-2", # match with the alpha2
    fill_color="GnBu",
    fill_opacity=1.0,
    line_opacity=0.3,
    nan_fill_color="white", # nans shall be left white
    legend_name="Percentage of Top 20 Streams By Domestic Artists",
    name="Countries by Number of Artists in Top 500 Streams",
).add_to(m)


fig = folium.Figure(width = 1000, height = 800)
fig.add_child(m)

fig.save("domestic_artists_map.html")
m



## Web Plot

In [ ]:
# remove global
weekly_chart_df_20_clean = weekly_chart_df_20_clean[weekly_chart_df_20_clean['country'] != 'Global'] 

In [ ]:
# get country from ISO
weekly_chart_df_20_clean["artist_country_full"] = (weekly_chart_df_20_clean["artist_country"].apply(convert_country))

In [ ]:
# Manually change top unknowns
update = {'Korea, Republic of': 'South Korea', 
          'Russian Federation': 'Russia',
          'Türkiye': 'Turkey',
          'Moldova, Republic of': 'Moldova',
          'Taiwan, Province of China': 'Taiwan',
          'Viet Nam': 'Vietnam',
          'Venezuela, Bolivarian Republic of': 'Venezuela'
}

web_df = weekly_chart_df_20_clean.copy()
web_df['artist_country_full'] = web_df['artist_country_full'].replace(update)


In [ ]:
# web_df

In [ ]:
from pyvis.network import Network
# https://pyvis.readthedocs.io/en/latest/

g = Network(height="800px", width="100%", notebook=True,bgcolor="white")

# add country nodes
countries = set(web_df["country"]).union(set(web_df["artist_country_full"]))
stream_totals = (web_df.groupby("country")["streams"].sum())

for c in countries:
    size = stream_totals.get(c, 1) / 1.8e6 # scale size by streams
    g.add_node(c, label=c, size=size, font={'size': 22})
    

# aggregate stream flows
flows = (
    web_df[web_df["artist_locality"] == "International"]
    .groupby(["country", "artist_country_full"])["streams"]
    .sum()
    .reset_index()
)

# add edges
for _, row in flows.iterrows():
    g.add_edge(
        row["country"],
        row["artist_country_full"],
        value=row["streams"]
    )

g.show("music_flows.html")


## Genre Map

In [ ]:
# group by genre and country
genre_df = (weekly_chart_df_20_clean.groupby(['country', 'artist_genre'])['streams'].sum().reset_index())
genre_df = genre_df[genre_df['artist_genre'] != 'Unknown']

In [ ]:
# obtain relative stream share by genre per country
genre_df['stream_pct'] = genre_df['streams'] / genre_df.groupby('country')['streams'].transform('sum') 


In [ ]:
# obtain highest
genre_df_first = (genre_df.sort_values('stream_pct', ascending=False).groupby('country').first()).reset_index()

In [ ]:
genre_df_first['artist_genre'].value_counts()

In [ ]:
# create manual groupings
genre_map = {'Hip Hop': 'Hip Hop', 'Pop Rap': 'Hip Hop',
    'Pop': 'Pop', 'Indie Pop': 'Pop', 'Indie-Pop': 'Pop', 'K-Pop': 'Pop', 'Q-Pop': 'Pop', 'Indonesian Pop': 'Pop', 'T-Pop': 'Pop', 
    'Cantopop': 'Pop', 'Alternative Pop': 'Pop',
    'Reggaeton': 'Latin', 'Latin Ballad': 'Latin', 'Regional Mexicano': 'Latin', 'Latin Urban': 'Latin',
    'Rock': 'Rock', 'J-Rock': 'Rock', 'Alternative Rock': 'Rock',
    'Filmi': 'Region Genre', 'Maskandi': 'Region Genre', 'Chalga': 'Region Genre', 'French': 'Region Genre',
    'Afrobeats': 'Other', 'Funk': 'Other', 'Jazz': 'Other', 'Psytrance': 'Other', 'Neo Soul': 'Other', 'Lo-Fi Hip Hop': 'Other', 
    'Indietronica': 'Other', 'Singer-Songwriter': 'Other'
}
             
                
genre_df_first['genre_main'] = genre_df_first['artist_genre'].map(genre_map)

In [ ]:
genre_df_first

In [ ]:
tmp = genre_df_first.groupby('country')['genre_main'].agg(lambda x: x.value_counts().index[0]).reset_index()
tmp2 = genre_df_first.groupby('country')['artist_genre'].agg(lambda x: ", ".join(x.unique()[:5])).reset_index()
tmp3 = tmp.merge(tmp2, on='country')


In [ ]:
import geopandas as gpd
import numpy as np

In [ ]:
# world map with data
url = 'https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/world-countries.json'
world_url = gpd.read_file(url)


In [ ]:
# merge df with geodata
world_merged = world_url.merge(tmp3, left_on='name', right_on='country', how='left')

In [ ]:
world_merged

In [ ]:
# fix US
world_merged.loc[world_merged['name'] == 'United States of America', 'country'] = 'United States'
world_merged.loc[world_merged['name'] == 'United States of America', 'genre_main'] = 'Latin'
world_merged.loc[world_merged['name'] == 'United States of America', 'artist_genre'] = 'Reggaeton'

In [ ]:
world_merged = world_merged.replace({np.nan: None})

In [ ]:
# colors 
color_map = {'Hip Hop': '#ff5959',
             'Pop': '#00a1ff',
             'Latin': '#ddff61',
             'Rock': '#00ff00',
             'Region Genre': '#ffa500',
             'Other': '#00ffff'           
}

In [ ]:
# plot genre maps, used seaborn because the text seemed cleaner than folium
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

colors = world_merged["genre_main"].map(lambda x: color_map.get(x, '#ffffff'))


# we'll use a function to get cleaner images for continent splits
def plot_genre_map(title, x_limits, y_limits, save):
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # plot
    world_merged.plot(
        ax=ax,
        color=colors,
        edgecolor="black",
        linewidth=0.5
    )
    
    # text
    for _, country in world_merged.iterrows():
        if country["geometry"] is not None and country["genre_main"] not in ["Unknown", None]:
            point = country["geometry"].representative_point()
            
            # Only label if the point is within our current zoom window
            if x_limits[0] <= point.x <= x_limits[1] and y_limits[0] <= point.y <= y_limits[1]:
                label_text = str(country["artist_genre"]).replace(", ", "\n")
                ax.text(point.x, point.y, label_text,
                    fontsize=6,ha="center",va="center")


    # legend
    legend_handles = [
        mpatches.Patch(color=color, label=genre) 
        for genre, color in color_map.items()
    ]
    
    # Add the legend to the plot
    ax.legend(
        handles=legend_handles, 
        title="Genre Groups", 
        loc='lower left', # You can change to 'lower right' if it blocks data
        bbox_to_anchor=(0.05, 0.05), # Fine-tune position
        fontsize=10,
        frameon=True,
        facecolor='white',
        edgecolor='grey'
    )
    
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
    ax.set_title(title, fontsize=15)
    ax.axis('off') 
    
    plt.tight_layout()
    plt.savefig(save)
    plt.show()




In [ ]:
# americas
plot_genre_map("", (-150, -30), (-57, 65), "map_americas.png")

In [ ]:
# europe
plot_genre_map("", (-20, 57), (-35, 75), "map_europe_africa.png")

In [ ]:
# asia
plot_genre_map("", (50, 152), (-40, 90), "map_asia.png")